**Unity Catalog Hosted SQL Functions with Databrics Agents**
![image_1780994469489.png](./image_1780994469489.png "image_1780994469489.png")

In [0]:
# Install utilities and Libraries 
%pip install databricks-langchain langchain-community langchain-experimental unitycatalog-ai[databricks] unitycatalog-langchain[databricks]

In [0]:
#restart the python kernel
dbutils.library.restartPython()

In [0]:
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0] 

print(catalog_name)

In [0]:
# Create a directory 'Unity Catalog hosted functions with Databricks (uc_hf_dbx)
dbutils.fs.mkdirs("/Volumes/dbx_apps_poc/mlpractice/genai_lab/uc_hf_dbx")

In [0]:
%run ../../utils/upload_data

In [0]:
# Import the file from the source to the unity catalog
# Define the currect catalog name
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0] 

# Define the file path
file_path = f"/Volumes/{catalog_name}/mlpractice/genai_lab/uc_hf_dbx"

# Define the file name
file_name = ["electronics_products.csv"]

# source file path / url
src_path = "https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/LangChain"

# Upload the file to the volume
upload_to_volume(src_path, file_name, file_path)

In [0]:
# Ingest the data from the csv file into a delta table in the unity catalog
from pyspark.sql import functions as F

df = (
    # 1. Read the file from the volume
    spark
        .readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", f"{file_path}/electronics_products_schema")
            .option("cloudFiles.schemaEvolutionMode", "rescue")
            .option("cloudFiles.inferColumnTypes", True)
            .load("/Volumes/dbx_apps_poc/mlpractice/genai_lab/uc_hf_dbx/")
            .withColumn("timestamp", F.current_timestamp())
            .withColumn("file_name", F.expr("_metadata.file_path"))
        # 2. Write the data to a delta table in the unity catalog
        .writeStream 
            .option("checkpointLocation", f"{file_path}/electronics_products_checkpoint")
            .option("mergeSchema", True)
            .trigger(availableNow=True)
            .table("dbx_apps_poc.rag.electronics_products")
) 

 

In [0]:
# Creating the unity catalog Hosted Functions Client
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()


In [0]:
%sql
-- Define the tool logic and registering in Unity Catalog
CREATE OR REPLACE FUNCTION rag.lookup_electronics_item(
    product_name STRING COMMENT 'Name of the product to lookup. For instance, if the user query is "How much does green webcam cost", then product name is "webcam" and not "green webcam"',
    product_colour STRING COMMENT 'Colour of the product to lookup.'
)
RETURNS STRING 
COMMENT 'Returns metadata about a specific product in the electronics_items dataset, including its ID, price, and description'
RETURN SELECT CONCAT(
    'product ID: ', productID, ', ',
    'Product Name: ', productName, ', ',
    'Product Colour: ', colour, ', ',
    'Product Price: ', price, ', '
)
FROM dbx_apps_poc.rag.electronics_products
WHERE LOWER(productName) = LOWER(product_name) AND LOWER(colour) = LOWER(product_colour)
LIMIT 1
 

In [0]:
# Creating the tool object for Usage within the Langchain Agent

from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit


# Create the toolkit with the unity catalog function
fun_name = "dbx_apps_poc.rag.lookup_electronics_item"
toolkit = UCFunctionToolkit(function_names=[fun_name])

tools = toolkit.tools


In [0]:
# Create the tool Calling Agent with Langchain
import mlflow
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# Patch: databricks-vectorsearch 0.74 renamed VectorSearchIndex to AISearchIndex
from databricks.vector_search import client as _vs_client
_vs_client.VectorSearchIndex = _vs_client.AISearchIndex

from databricks_langchain import ChatDatabricks, UCFunctionToolkit
 

# Initialize the LLM
LLM_ENDPOINT_NAME = "databricks-claude-haiku-4-5"
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT_NAME,
    temperature=0.1,
    #max_tokens=1000,
    #verbose=True,
)

# Define the prompt with agent_scratchpad placeholder 
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Make sure to use tools for additional functionality"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Enable automatic tracing 
mlflow.langchain.autolog()

# Define the agent, specifying the tools form the toolkit and above
agent = create_tool_calling_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent,
    tools=tools,
    verbose=True,
) 

In [0]:
# Test and Invok the agent with agent excuter 
agent_executor.invoke({"input": "Return details for the Red Graphics Card"})

In [0]:
agent_executor.invoke({"input": "Return details for the blue keyboard"})